# Feature Extraction — In-the-Wild Deepfake Speech Dataset
Extracts rf_features (MFCC mean + std), 1dcnn_features (MFCC matrix), and hybrid_features (mel spectrogram).

**Uses CPU runtime only**

Author — Claudia Pletka

In [1]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
!pip install -q numpy pandas soundfile librosa tqdm scikit-learn

In [2]:
# ── Cell 2: Imports ───────────────────────────────────────────────────────────
import os
import gc
import json

import numpy as np
import pandas as pd
import soundfile as sf
import librosa
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from google.colab import drive, files

In [3]:
# ── Cell 3: Confirm CPU runtime ───────────────────────────────────────────────
import torch
if torch.cuda.is_available():
    print(f'WARNING: You are on a GPU ({torch.cuda.get_device_name(0)}). '
          'Switch to CPU runtime to save compute units!')
else:
    print('CONFIRMED: CPU-only runtime. Good to go.')

CONFIRMED: CPU-only runtime. Good to go.


In [4]:
# ── Cell 4: Mount Drive ───────────────────────────────────────────────────────
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [5]:
# ── Cell 5: Download & unzip In-the-Wild dataset (official Hugging Face mirror) ──

print('Downloading In-the-Wild dataset from Hugging Face... (~7-8 GB, may take a few minutes)')
!wget -q --show-progress https://huggingface.co/datasets/mueller91/In-The-Wild/resolve/main/release_in_the_wild.zip -O /content/release_in_the_wild.zip

print('\nUnzipping...')
!unzip -q /content/release_in_the_wild.zip -d /content/in_the_wild/

# Count audio files
audio_exts = ('.wav', '.flac', '.mp3', '.WAV', '.FLAC')
count = 0
for root, dirs, files_ in os.walk('/content/in_the_wild/'):
    for f in files_:
        if f.endswith(audio_exts):
            count += 1
print(f'\nDone! Total audio files found: {count}')

print('\n── Top-level contents ──')
!ls /content/in_the_wild/

/content/release_in 100%[===================>]   7.60G  23.2MB/s    in 5m 44s  

Unzipping...

Done! Total audio files found: 31779

── Top-level contents ──
release_in_the_wild


In [6]:
# ── Cell 6: Inspect dataset structure ─────────────────────────────────────────
print('── Folder structure (top 2 levels) ──────────────────────')
for root, dirs, files_ in os.walk('/content/in_the_wild/'):
    depth = root.replace('/content/in_the_wild/', '').count(os.sep)
    if depth < 2:
        indent = '  ' * depth
        print(f'{indent}{os.path.basename(root)}/')
        audio_files = [f for f in files_ if f.endswith(('.wav', '.flac'))]
        if audio_files:
            print(f'{indent}  [{len(audio_files)} audio files — e.g. {audio_files[0]}]')

print('\n── Metadata CSV ──────────────────────────────────────────')
for root, dirs, files_ in os.walk('/content/in_the_wild/'):
    for f in files_:
        if f.endswith('.csv'):
            csv_path = os.path.join(root, f)
            print(f'Found CSV : {csv_path}')
            preview = pd.read_csv(csv_path)
            print(f'Columns   : {preview.columns.tolist()}')
            print(f'Shape     : {preview.shape}')
            print(f'Labels    : {preview.iloc[:, -1].value_counts().to_dict()}')
            print(preview.head())

── Folder structure (top 2 levels) ──────────────────────
/
release_in_the_wild/
  [31779 audio files — e.g. 6707.wav]

── Metadata CSV ──────────────────────────────────────────
Found CSV : /content/in_the_wild/release_in_the_wild/meta.csv
Columns   : ['file', 'speaker', 'label']
Shape     : (31779, 3)
Labels    : {'bona-fide': 19963, 'spoof': 11816}
    file               speaker      label
0  0.wav         Alec Guinness      spoof
1  1.wav         Alec Guinness      spoof
2  2.wav          Barack Obama      spoof
3  3.wav         Alec Guinness      spoof
4  4.wav  Christopher Hitchens  bona-fide


In [7]:
# ── Cell 7: Configure paths (In-the-Wild version) ─────────────────────────────
BASE_DIR = '/content/in_the_wild/release_in_the_wild'
META_CSV = os.path.join(BASE_DIR, 'meta.csv')

# Output pkl files saved to Drive (v2 = In-the-Wild, to avoid overwriting FoR results)
TRAIN_OUT = '/content/drive/MyDrive/ITW_Features_train_v2.pkl'
DEV_OUT   = '/content/drive/MyDrive/ITW_Features_dev_v2.pkl'
EVAL_OUT  = '/content/drive/MyDrive/ITW_Features_eval_v2.pkl'

# Sanity check
print(f'Base dir   : {BASE_DIR}')
print(f'Meta CSV   : {META_CSV} (exists: {os.path.exists(META_CSV)})')
audio_count = sum(1 for f in os.listdir(BASE_DIR) if f.endswith('.wav'))
print(f'Audio files: {audio_count}')

Base dir   : /content/in_the_wild/release_in_the_wild
Meta CSV   : /content/in_the_wild/release_in_the_wild/meta.csv (exists: True)
Audio files: 31779


---
## Step 2 — Build train / dev / eval splits

In [8]:
# ── Cell 8: Build train/dev/eval splits stratified by speaker ─────────────────
# In-the-Wild contains the same speaker across many clips, so a random row-level
# split would leak the same celebrity into both train and test. We split SPEAKERS
# instead, so each speaker appears in only one of train/dev/eval.

# Load the metadata
meta = pd.read_csv(META_CSV)
print(f'Total samples  : {len(meta)}')
print(f'Label counts   : {meta["label"].value_counts().to_dict()}')
print(f'Unique speakers: {meta["speaker"].nunique()}')

# Map labels: bona-fide -> 0, spoof -> 1 (same convention used previously)
meta['label_int'] = meta['label'].map({'bona-fide': 0, 'spoof': 1})

# Build full path to each audio file
meta['full_path'] = meta['file'].apply(lambda f: os.path.join(BASE_DIR, f))
meta['file_id']   = meta['file']

# Speaker-disjoint split: 70% train / 15% dev / 15% eval
np.random.seed(42)
all_speakers = meta['speaker'].unique().copy()
np.random.shuffle(all_speakers)

n_speakers = len(all_speakers)
n_train = int(n_speakers * 0.70)
n_dev   = int(n_speakers * 0.15)

train_speakers = set(all_speakers[:n_train])
dev_speakers   = set(all_speakers[n_train:n_train + n_dev])
eval_speakers  = set(all_speakers[n_train + n_dev:])

train_meta = meta[meta['speaker'].isin(train_speakers)][['file_id', 'full_path', 'speaker']].copy()
train_meta['label'] = meta.loc[train_meta.index, 'label_int']

dev_meta = meta[meta['speaker'].isin(dev_speakers)][['file_id', 'full_path', 'speaker']].copy()
dev_meta['label'] = meta.loc[dev_meta.index, 'label_int']

eval_meta = meta[meta['speaker'].isin(eval_speakers)][['file_id', 'full_path', 'speaker']].copy()
eval_meta['label'] = meta.loc[eval_meta.index, 'label_int']

print(f'\nSpeakers — Train: {len(train_speakers)} | Dev: {len(dev_speakers)} | Eval: {len(eval_speakers)}')
print(f'Train: {len(train_meta):>5} samples | labels {train_meta["label"].value_counts().to_dict()}')
print(f'Dev  : {len(dev_meta):>5} samples | labels {dev_meta["label"].value_counts().to_dict()}')
print(f'Eval : {len(eval_meta):>5} samples | labels {eval_meta["label"].value_counts().to_dict()}')

# Verify no speaker overlap between splits
assert not (train_speakers & dev_speakers),  'Speaker leak: train ∩ dev'
assert not (train_speakers & eval_speakers), 'Speaker leak: train ∩ eval'
assert not (dev_speakers & eval_speakers),   'Speaker leak: dev ∩ eval'
print('\nNo speaker overlap between splits — good.')

Total samples  : 31779
Label counts   : {'bona-fide': 19963, 'spoof': 11816}
Unique speakers: 54

Speakers — Train: 37 | Dev: 8 | Eval: 9
Train: 26878 samples | labels {0: 17135, 1: 9743}
Dev  :  2485 samples | labels {0: 1247, 1: 1238}
Eval :  2416 samples | labels {0: 1581, 1: 835}

No speaker overlap between splits — good.


---
## Step 3 — Feature Extraction

In [9]:
# ── Cell 9: Feature extraction function ──────────────────────────────────────
def extract_all_features_from_samples(file_path, n_mfcc=40, n_mels=128):
    """
    Returns three feature representations:
      rf_features     : (80,)     MFCC mean + std concatenated
      1dcnn_features  : (T, 40)   full MFCC matrix transposed
      hybrid_features : (T, 128)  mel spectrogram in dB transposed
    """
    audio, sr = sf.read(file_path)

    # Convert stereo to mono if needed
    if audio.ndim > 1:
        audio = np.mean(audio, axis=1)

    # Resample to 16kHz to ensure consistent frame counts across all files
    if sr != 16000:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=16000)
        sr = 16000

    # Fixed 4-second length — pad short, trim long
    target_len = sr * 4
    if len(audio) > target_len:
        audio = audio[:target_len]
    else:
        audio = np.pad(audio, (0, target_len - len(audio)), mode='constant')

    # ── MFCC ──────────────────────────────────────────────────────────────────
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=n_mfcc)  # (40, T)

    # rf_features: mean + std across time → (80,)
    mfcc_mean   = np.mean(mfcc, axis=1)                 # (40,)
    mfcc_std    = np.std(mfcc,  axis=1)                 # (40,)
    rf_features = np.concatenate([mfcc_mean, mfcc_std]) # (80,)

    # 1dcnn_features: full matrix → (T, 40)
    dcnn_features = mfcc.T

    # ── Mel spectrogram ───────────────────────────────────────────────────────
    mels            = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=n_mels)
    spec_db         = librosa.power_to_db(mels, ref=np.max)   # (128, T)
    hybrid_features = spec_db.T                                # (T, 128)

    return rf_features, dcnn_features, hybrid_features

In [10]:
# ── Cell 10: Extraction pipeline function ─────────────────────────────────────
def feature_extraction_sprint(split_meta, save_path, split_name):
    processed_data = []

    for _, row in tqdm(split_meta.iterrows(), total=len(split_meta),
                       desc=f'Processing {split_name}'):
        try:
            f_rf, f_1dcnn, f_hybrid = extract_all_features_from_samples(row['full_path'])
            processed_data.append({
                'file_id'        : row['file_id'],
                'label'          : row['label'],
                'rf_features'    : f_rf,
                '1dcnn_features' : f_1dcnn,
                'hybrid_features': f_hybrid
            })
        except Exception as e:
            print(f'  [!] Skipped {row["file_id"]}: {e}')
            continue

    if processed_data:
        df = pd.DataFrame(processed_data)
        df.to_pickle(save_path)
        print(f'\nSUCCESS: {split_name.upper()} ({len(df)} samples) saved to {save_path}')
        del processed_data, df
        gc.collect()
    else:
        print(f'\n[!] No samples processed for {split_name}.')

In [11]:
# ── Cell 11: Run extraction — Train ──────────────────────────────────────────
feature_extraction_sprint(train_meta, TRAIN_OUT, 'train')

Processing train: 100%|██████████| 26878/26878 [10:55<00:00, 41.03it/s]



SUCCESS: TRAIN (26878 samples) saved to /content/drive/MyDrive/ITW_Features_train_v2.pkl


In [12]:
# ── Cell 12: Run extraction — Dev ────────────────────────────────────────────
feature_extraction_sprint(dev_meta, DEV_OUT, 'dev')

Processing dev: 100%|██████████| 2485/2485 [01:07<00:00, 36.86it/s]



SUCCESS: DEV (2485 samples) saved to /content/drive/MyDrive/ITW_Features_dev_v2.pkl


In [13]:
# ── Cell 13: Run extraction — Eval ───────────────────────────────────────────
feature_extraction_sprint(eval_meta, EVAL_OUT, 'eval')

Processing eval: 100%|██████████| 2416/2416 [00:59<00:00, 40.60it/s]



SUCCESS: EVAL (2416 samples) saved to /content/drive/MyDrive/ITW_Features_eval_v2.pkl


---
## Step 4 — Verify Output

In [14]:
# ── Cell 14: Verify output ───────────────────────────────────────────────────
for name, path in [('Train', TRAIN_OUT), ('Dev', DEV_OUT), ('Eval', EVAL_OUT)]:
    df = pd.read_pickle(path)
    print(f'\n── {name} ────────────────────────────────────────────')
    print(f'  Samples : {len(df)}')
    print(f'  Labels  : {df["label"].value_counts().to_dict()}')
    print(f'  rf_features     shape: {df["rf_features"].iloc[0].shape}')      # expect (80,)
    print(f'  1dcnn_features  shape: {df["1dcnn_features"].iloc[0].shape}')   # expect (T, 40)
    print(f'  hybrid_features shape: {df["hybrid_features"].iloc[0].shape}')  # expect (T, 128)
    del df
    gc.collect()


── Train ────────────────────────────────────────────
  Samples : 26878
  Labels  : {0: 17135, 1: 9743}
  rf_features     shape: (80,)
  1dcnn_features  shape: (126, 40)
  hybrid_features shape: (126, 128)

── Dev ────────────────────────────────────────────
  Samples : 2485
  Labels  : {0: 1247, 1: 1238}
  rf_features     shape: (80,)
  1dcnn_features  shape: (126, 40)
  hybrid_features shape: (126, 128)

── Eval ────────────────────────────────────────────
  Samples : 2416
  Labels  : {0: 1581, 1: 835}
  rf_features     shape: (80,)
  1dcnn_features  shape: (126, 40)
  hybrid_features shape: (126, 128)
